# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. One row = one pseudonymized content item (`content_id`).
2. The dataset used is `data/raw/content_refresh_anonymized.csv` (30,000 rows × 44 columns).
3. The data covers trailing 90-day metrics for each content item (e.g., `impressions_90d`).
4. The prediction target is `is_declining`, a binary label derived as `1` if `trend_direction == 'down'`, else `0`.
5. `trend_direction` and `trend_pct` are excluded due to label leakage (used to derive `is_declining`), and `content_id`/`client_id` are excluded as features (pseudonyms).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Classification

-   **Label / proxy**
    -   `is_declining`: A derived binary target (`1` if `trend_direction == 'down'`, else `0`).

-   **Context**
    -   `content_id`: Pseudonym for grouping/splitting only; not a feature.
    -   `client_id`: Pseudonym for grouping/splitting only; not a feature.

-   **Excluded**
    -   `trend_direction`: Excluded due to label leakage (directly used to derive `is_declining`).
    -   `trend_pct`: Excluded due to label leakage (informs `trend_direction`, which derives `is_declining`).

-   **Features**
    -   `search_volume`
    -   `competition`
    -   `competition_level`
    -   `cpc`
    -   `content_type`
    -   `main_intent`
    -   `word_count`
    -   `char_count`
    -   `provider_used`
    -   `model_used`
    -   `impressions_90d`
    -   `clicks_90d`
    -   `pageviews_90d`
    -   `sessions_90d`
    -   `users_90d`
    -   `engaged_sessions_90d`
    -   `ai_sessions_90d`
    -   `scroll_events_90d`
    -   `days_with_impressions`
    -   `days_with_sessions`
    -   `impressions_last_30d`
    -   `clicks_last_30d`
    -   `sessions_last_30d`
    -   `impressions_prev_30d`
    -   `clicks_prev_30d`
    -   `sessions_prev_30d`
    -   `content_age_days`
    -   `age_tier`
    -   `age_tier_order`
    -   `days_since_last_update`
    -   `freshness_tier`
    -   `word_count_tier`
    -   `char_count_tier`
    -   `ctr`
    -   `avg_position`
    -   `engagement_rate`
    -   `scroll_rate`
    -   `ai_traffic_pct`
    -   `impression_tier`
    -   `position_tier`

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
import pandas as pd
try:
    df = pd.read_csv('content_refresh_anonymized.csv')
    print('Dataset loaded successfully.')
except FileNotFoundError:
    print('Error: The file data/raw/content_refresh_anonymized.csv was not found. Please ensure it is in the correct path.')
    df = None # Set df to None to prevent further errors if file is not found

if df is not None:
    print(f'Total number of rows: {len(df)}')
    display(df.head())

Dataset loaded successfully.
Total number of rows: 30000


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


### 3.1 Verify Unit of Analysis (Grain)

We stated that 'one row = one pseudonymized content item'. Let's verify this by checking the uniqueness of `content_id`.

In [3]:
if df is not None:
    unique_content_ids = df['content_id'].nunique()
    total_rows = len(df)
    print(f'Number of unique `content_id` values: {unique_content_ids}')
    print(f'Total number of rows: {total_rows}')

    if unique_content_ids == total_rows:
        print('Verification successful: Each row represents a unique content item (`content_id` is unique).')
    else:
        print('Verification failed: `content_id` is not unique, indicating that a row might not always represent a single content item.')

Number of unique `content_id` values: 30000
Total number of rows: 30000
Verification successful: Each row represents a unique content item (`content_id` is unique).


### 3.2 Verify Missing Values

Let's check for missing values across all columns to ensure data quality and identify potential issues.

In [4]:
if df is not None:
    missing_values = df.isnull().sum()
    missing_percentage = (df.isnull().sum() / len(df)) * 100
    missing_df = pd.DataFrame({'Missing Count': missing_values, 'Missing Percentage': missing_percentage})
    missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)

    if not missing_df.empty:
        print('Columns with missing values:')
        display(missing_df)
    else:
        print('No missing values found in any column.')

Columns with missing values:


,Missing Count,Missing Percentage
provider_used,21438,71.460000
word_count_tier,7699,25.663333
char_count,7699,25.663333
word_count,7699,25.663333
char_count_tier,7699,25.663333
model_used,5733,19.110000
trend_pct,3388,11.293333
competition_level,2610,8.700000
search_volume,2468,8.226667
competition,2468,8.226667


### 3.3 Verify Time Windows (Trailing 90-day metrics)

We claimed the data covers 'trailing 90-day metrics'. While we don't have exact timestamps for the metrics, we can check the `content_age_days` and `days_since_last_update` columns to understand the time distribution. The `is_declining_label` implies a comparison over a time window, which we will assume is handled upstream.

In [5]:
if df is not None:
    # Check distribution of content_age_days
    print('Distribution of `content_age_days`:')
    display(df['content_age_days'].describe())

    # Check distribution of days_since_last_update
    print('\nDistribution of `days_since_last_update`:')
    display(df['days_since_last_update'].describe())

    # Check the range of other time-related metrics if available (e.g., if there were explicit date columns)
    # For now, we rely on the describe output to infer the 'freshness' of the content relative to the observation date.

Distribution of `content_age_days`:


,content_age_days
count,30000.00000
mean,256.16780
std,132.70793
min,90.00000
25%,132.00000
50%,236.00000
75%,333.00000
max,564.00000



Distribution of `days_since_last_update`:


,days_since_last_update
count,30000.000000
mean,46.098300
std,42.078709
min,1.000000
25%,20.000000
50%,20.000000
75%,104.000000
max,373.000000


### 3.4 Verify Prediction Target and Exclusions

We identified `is_declining_label` as the binary classification target and `trend_direction`, `trend_pct` as excluded due to label leakage. Let's confirm `is_declining_label` is binary and that the excluded columns exist.

In [7]:
if df is not None:
    # Print all columns to debug the KeyError
    print('Columns in DataFrame:', df.columns.tolist())

    # Check `is_declining_label` values
    print('Unique values in `is_declining_label`:')
    try:
        display(df['is_declining_label'].value_counts())
    except KeyError:
        print("Error: 'is_declining_label' column not found. Please check the dataset or column name.")

    # Confirm existence of excluded columns
    excluded_cols = ['trend_direction', 'trend_pct']
    print('\nChecking for excluded columns:')
    for col in excluded_cols:
        if col in df.columns:
            print(f'Column `{col}` exists in the dataset.')
        else:
            print(f'Column `{col}` does NOT exist in the dataset. This might be an error in the contract.')

Columns in DataFrame: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Unique values in `is_declining_label`:
Error: 'is_declining_label' column not found. Please check the dataset or column name.

Checking for excluded columns:
Column `trend_direction` exists in the dataset.
C

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### 4. Data Limits

This dataset, while valuable, has inherent limitations that restrict the types of questions it can answer:

1.  **Limited Historical Depth**: The metrics are aggregated over a 'trailing 90-day' window. This means we cannot analyze trends or performance over longer periods (e.g., year-over-year changes) or for specific events outside this window. We also lack the granular daily raw data within the 90-day window, making detailed intra-window time-series analysis impossible.
2.  **Inferred Data Source Limitations**: Given column names like `impressions_90d` and `avg_position`, the data likely originates from search performance tools (e.g., GSC). This means it may not capture performance from other crucial channels like social media, direct traffic, or email campaigns, limiting a holistic view of content performance.
3.  **Pseudonymization**: While essential for privacy, `content_id` and `client_id` prevent linking content items to actual client names or detailed business contexts. This restricts analyses that require real-world client identification or specific content details.
4.  **Missing Values and Bias**: Significant missingness in columns like `provider_used` (71.46%), `word_count` (25.66%), `char_count` (25.66%), and `model_used` (19.11%) could lead to biased models if not handled carefully. Analysis of these features may not represent the full content population.
5.  **Derived Label Abstraction**: The `is_declining` label is derived directly from `trend_direction`. This means our model will predict a categorical 'down' trend rather than directly inferring a broader, more nuanced concept of 'content decline' that might involve other factors not captured by `trend_direction`.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.